# Simple Neural Network (Multilayer Perceptron)

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import mlflow

/opt/miniconda3/envs/msds/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv('../train_competition_2026.csv')
train_df.head()

,obs,sub_id,time,num_0,num_1,num_2,cat_0,cat_1,cat_2,cat_3,cat_4,t_0,t_1,t_2,t_3,t_4,y_1,y_2
0,0,0,2068-09-19 23:34:11,1.38,49,7,1,3,1,0,1,105.5,95.0,67.4,36.6,23.2,33.4,107.4
1,0,0,2068-09-19 23:35:11,1.38,49,7,1,3,1,0,1,104.4,95.0,66.4,37.8,22.7,33.4,107.4
2,0,0,2068-09-19 23:36:11,1.38,49,7,1,3,1,0,1,104.0,95.0,65.2,37.0,22.1,33.4,107.4
3,0,0,2068-09-19 23:37:11,1.38,49,7,1,3,1,0,1,102.8,95.0,63.4,35.9,20.7,33.4,107.4
4,0,0,2068-09-19 23:38:11,1.38,49,7,1,3,1,0,1,101.3,95.1,59.1,34.5,18.1,33.4,107.4


In [3]:
train_df['time'] = pd.to_datetime(train_df['time'], format='%Y-%m-%d %H:%M:%S')

In [4]:
test_df = pd.read_csv('../test_no_outcome.csv')
test_df.head()

,obs,sub_id,time,num_0,num_1,num_2,cat_0,cat_1,cat_2,cat_3,cat_4,t_0,t_1,t_2,t_3,t_4
0,18,1,2134-04-01 22:23:14,-1.0,38,1,1,1,0,0,0,105.4,99.8,50.7,61.4,36.8
1,18,1,2134-04-01 22:24:14,-1.0,38,1,1,1,0,0,0,105.4,99.4,49.4,61.1,36.2
2,18,1,2134-04-01 22:25:14,-1.0,38,1,1,1,0,0,0,104.6,99.0,49.7,61.4,36.6
3,18,1,2134-04-01 22:26:14,-1.0,38,1,1,1,0,0,0,104.5,99.6,51.7,61.8,37.2
4,18,1,2134-04-01 22:27:14,-1.0,38,1,1,1,0,0,0,104.6,99.5,52.5,61.9,37.5


## MLP with PCA

In [5]:
def flatten_blocks(df, feature_cols):
    X_flat = df.groupby(["obs"])[feature_cols].apply(lambda block: block.values.flatten())
    X_flat = pd.DataFrame(X_flat.tolist(), index=X_flat.index)
    return X_flat.reset_index()

In [6]:
feature_cols = [c for c in train_df.columns if c not in ["sub_id", "obs", "time", "y_1", "y_2"]]

X_train_flat = flatten_blocks(train_df, feature_cols)
X_test_flat  = flatten_blocks(test_df, feature_cols)

In [7]:
X_train_flat.head()

,obs,0,1,2,3,4,5,6,7,8,...,380,381,382,383,384,385,386,387,388,389
0,0,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,105.5,...,1.0,3.0,1.0,0.0,1.0,94.5,93.8,59.8,34.3,17.7
1,1,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,98.6,...,1.0,3.0,1.0,0.0,1.0,95.3,92.9,52.1,38.0,22.0
2,2,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,93.1,...,1.0,3.0,1.0,0.0,1.0,91.2,93.4,52.5,36.1,19.1
3,3,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,85.1,...,1.0,3.0,1.0,0.0,1.0,87.1,95.0,61.8,36.4,20.8
4,4,1.38,49.0,7.0,1.0,3.0,1.0,0.0,1.0,88.7,...,1.0,3.0,1.0,0.0,1.0,86.1,89.6,66.7,32.5,17.8


In [8]:
def extract_targets(df):
    return df.groupby(["obs"])[["y_1", "y_2"]].first().reset_index(drop=True)

y_train = extract_targets(train_df)

In [9]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, output_dim, num_layers=1, hidden_size=32, dropout=0.0):
        super().__init__()
        
        layers = []
        
        # Input layer
        layers.append(nn.Linear(input_dim, hidden_size))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        
        # Hidden layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        
        # Output layer
        layers.append(nn.Linear(hidden_size, output_dim))
        
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [10]:
def train_mlp(
    X_train, y_train, X_val, y_val,
    input_dim, output_dim, num_layers=1,
    hidden_size=32, dropout=0.0, weight_decay=1e-4,
    epochs=500, epoch_int=10, patience=20, lr=1e-3,
):
    
    model = SimpleMLP(
        input_dim=input_dim,
        output_dim=output_dim,
        num_layers=num_layers,
        hidden_size=hidden_size,
        dropout=dropout
    ).to("cpu")

    criterion = nn.MSELoss()  # change for classification if needed
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Convert to tensors
    X_train = torch.tensor(X_train, dtype=torch.float32).to("cpu")
    y_train = torch.tensor(y_train, dtype=torch.float32).to("cpu")
    X_val   = torch.tensor(X_val, dtype=torch.float32).to("cpu")
    y_val   = torch.tensor(y_val, dtype=torch.float32).to("cpu")

    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(epochs):
        # Train
        model.train()
        optimizer.zero_grad()

        train_pred = model(X_train)
        train_loss = criterion(train_pred, y_train)

        train_loss.backward()
        optimizer.step()

        # Val
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val)
            
            # Calc r2
            train_r2 = r2_score(y_train.cpu().numpy(), train_pred.detach().cpu().numpy())
            val_r2 = r2_score(y_val.cpu().numpy(), val_pred.cpu().numpy())

        if epoch % epoch_int == 0:
            print(f'Epoch {epoch:03d} | '
                f'Train Loss: {train_loss.item():.4f} | Val Loss {val_loss.item():.4f} | '
                f'Train R2: {train_r2:.4f} | Val R2 {val_r2:.4f}')

        # Early stopping
        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    # Restore best model
    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        final_val_pred = model(X_val)
        final_val_loss = criterion(final_val_pred, y_val)

    return model, final_val_loss

In [11]:
def run_pipeline(hp):
    k = hp['pca_k']
    
    scaler_x = StandardScaler()
    X_train_raw = X_train_flat.drop(columns=["obs"])
    
    X_train_scaled = scaler_x.fit_transform(X_train_raw)

    pca = PCA(n_components=k, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    
    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train)

    X_train_split, X_val, y_train_split, y_val = train_test_split(
        X_train_pca, y_train_scaled, test_size=0.2, random_state=42
    )

    model, val_loss = train_mlp(
        X_train_split, y_train_split, X_val, y_val,
        input_dim=X_train_split.shape[1],
        output_dim=2,
        num_layers=hp['num_layers'],
        hidden_size=hp['hidden_size'],
        dropout=hp['dropout'],
        weight_decay=hp['weight_decay'],
        epochs=hp['epochs'],
        epoch_int=hp['epoch_int'],
        patience=hp['patience'],
        lr=hp['lr']
    )
    
    return model, val_loss, scaler_x, scaler_y, pca

In [12]:
hp = {
    'pca_k': 50,
    'hidden_size': 64,
    'num_layers': 4,
    'dropout': 0.0,
    'weight_decay': 1e-3,
    'epochs': 10000, 
    'epoch_int': 50,
    'patience': 50,
    'lr': 1e-3
}

model, val_loss, scaler_x, scaler_y, pca = run_pipeline(hp)

Epoch 000 | Train Loss: 1.0171 | Val Loss 0.9767 | Train R2: -0.0119 | Val R2 0.0025
Epoch 050 | Train Loss: 0.2168 | Val Loss 0.2184 | Train R2: 0.7840 | Val R2 0.7780
Epoch 100 | Train Loss: 0.1921 | Val Loss 0.1967 | Train R2: 0.8086 | Val R2 0.8002
Epoch 150 | Train Loss: 0.1846 | Val Loss 0.1946 | Train R2: 0.8160 | Val R2 0.8023
Epoch 200 | Train Loss: 0.1774 | Val Loss 0.1958 | Train R2: 0.8233 | Val R2 0.8010
Early stopping at epoch 201


In [13]:
def get_real_predictions(model, X_test_df, scaler_x, scaler_y, pca):
    X_raw = X_test_df.drop(columns=["obs"]) if "obs" in X_test_df.columns else X_test_df
    X_scaled = scaler_x.transform(X_raw)
    
    X_pca = pca.transform(X_scaled)
    
    model.eval()
    X_tensor = torch.FloatTensor(X_pca)
    
    with torch.no_grad():
        y_pred_tensor = model(X_tensor)

    y_pred_scaled = y_pred_tensor.cpu().numpy()
    y_pred_final = scaler_y.inverse_transform(y_pred_scaled)
    
    return y_pred_final

In [14]:
predictions = get_real_predictions(model, X_test_flat, scaler_x, scaler_y, pca)

y_test_pred = get_real_predictions(model, X_test_flat, scaler_x, scaler_y, pca)

pred_df = pd.concat(
    [
        X_test_flat[["obs"]].reset_index(drop=True), 
        pd.DataFrame(y_test_pred, columns=["y_1", "y_2"])
    ],
    axis=1
)

len(y_test_pred)

3450

In [15]:
pred_df.to_csv("mlp_pca_1.csv", index=False)

In [ ]:
def run_with_mlflow(hp):
    run_name = f"lstm_k{hp['pca_k']}_h{hp['hidden_size']}_l{hp['num_layers']}_d{hp['dropout']}_w{hp['weight_decay']}"
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("tuning", "pca_v_hiddensize_2")
        mlflow.log_params(hp)
        model, val_loss, _, _, _ = run_pipeline(hp)
        mlflow.log_metric("val_loss", val_loss)
        
        return model, val_loss

if __name__ == '__main__':
    db_path = "/Users/connoryablonski/Dropbox/USF_MSDS/advanced_ml/mlflow.db"
    mlflow.set_tracking_uri(f"sqlite:///{db_path}")
    mlflow.set_experiment("AML_Project_MLP_Grid_Search")
    
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    print(f"Active Experiment: {mlflow.get_experiment_by_name('AML_Project_MLP_Grid_Search').experiment_id}")

    from itertools import product

    best_score = float("inf")
    
    pca_ks = [30, 40, 50, 60, 70, 80, 90, 100]
    hidden_sizes = [16, 32, 64, 128, 256]
    num_layers = [3]
    dropouts = [0.2]
    weight_decays = [1e-3]
    
    best_model = None
    best_pca = None

    for ks, hs, nl, dr, wd in product(pca_ks, hidden_sizes, num_layers, dropouts, weight_decays):
        
        hp = {
            'pca_k': ks,
            'hidden_size': hs,
            'num_layers': nl,
            'dropout': dr,
            'weight_decay': wd,
            'epochs': 10000, 
            'epoch_int': 50,
            'patience': 50,
            'lr': 5e-3
        }
        
        model, score = run_with_mlflow(hp)
        
        if score < best_score:
            best_score = score
            best_hp = hp
            best_model = model
            
            best_pca = PCA(n_components=hp['pca_k'], random_state=42)
            best_pca.fit(X_train_flat.drop(columns=["obs"]))
            

    print("Best:", best_hp, best_score)

Tracking URI: sqlite:////Users/connoryablonski/Dropbox/USF_MSDS/advanced_ml/mlflow.db
Active Experiment: 1
Epoch 000 | Train Loss: 1.2275 | Val Loss 1.1118 | Train R2: -0.2220 | Val R2 -0.1331
Epoch 050 | Train Loss: 0.2142 | Val Loss 0.2138 | Train R2: 0.7866 | Val R2 0.7827
Epoch 100 | Train Loss: 0.1997 | Val Loss 0.2009 | Train R2: 0.8010 | Val R2 0.7958
Epoch 150 | Train Loss: 0.1968 | Val Loss 0.1991 | Train R2: 0.8039 | Val R2 0.7977
Epoch 200 | Train Loss: 0.1952 | Val Loss 0.1981 | Train R2: 0.8055 | Val R2 0.7987
Epoch 250 | Train Loss: 0.1939 | Val Loss 0.1976 | Train R2: 0.8068 | Val R2 0.7992
Epoch 300 | Train Loss: 0.1925 | Val Loss 0.1970 | Train R2: 0.8081 | Val R2 0.7998
Epoch 350 | Train Loss: 0.1913 | Val Loss 0.1963 | Train R2: 0.8094 | Val R2 0.8005
Epoch 400 | Train Loss: 0.1904 | Val Loss 0.1964 | Train R2: 0.8103 | Val R2 0.8004
Early stopping at epoch 405
Epoch 000 | Train Loss: 1.1260 | Val Loss 0.9436 | Train R2: -0.1207 | Val R2 0.0373
Epoch 050 | Train Loss